In [1]:
%pip install pandas spacy transformers torch


[notice] A new release of pip is available: 25.1.1 -> 26.0
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Doğrudan mevcut kernel üzerinden modeli indiriyoruz
import sys
!{sys.executable} -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)

[notice] A new release of pip is available: 25.1.1 -> 26.0
[notice] To update, run: pip3.11 install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
import pandas as pd
import spacy
from transformers import pipeline

# ==========================================
# 1. SETUP MODELS
# ==========================================
print("Loading AI Models... (This might take a minute)")

try:
    nlp_ner = spacy.load("en_core_web_sm")
except OSError:
    print("Spacy model not found. Run: python -m spacy download en_core_web_sm")
    exit()

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
emotion_analyzer = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=False)

# ==========================================
# 2. ANALYSIS FUNCTIONS (IMPROVED)
# ==========================================

def get_manual_override(text):
    """
    Modelin 'Fire' kelimesini yanlış anlamasını önlemek için 
    anahtar kelime tabanlı ön kontrol.
    """
    text_lower = text.lower()
    
    # Silahlı çatışma ve soygun anahtar kelimeleri
    shooting_keywords = [
        'ak-47', 'gunfire', 'shots fired', 'officer down', '211', 
        'shooting', 'armed suspect', 'body armor', 'automatic weapon'
    ]
    
    # Eğer bu kelimeler geçiyorsa, 'Armed Robbery / Shooting' kategorisini zorla
    if any(kw in text_lower for kw in shooting_keywords):
        return "Armed Robbery / Shooting"
    
    return None

def analyze_incident_type(text):
    """ Determine the category of the emergency with context awareness """
    
    # Önce manuel kontrol yap (Modelin hatasını engellemek için)
    override = get_manual_override(text)
    if override:
        return override

    # Etiketleri daha spesifik hale getirdik
    # 'Fire Hazard' yerine 'Fire / Structural Fire' kullanarak karışıklığı azalttık
    labels = [
        "Medical Emergency", 
        "Fire or Smoke Incident", 
        "Armed Robbery / Shooting", 
        "Traffic Accident", 
        "Domestic Violence", 
        "Suspicious Activity"
    ]
    
    # Modelin 'fire' (ateş açmak) ve 'fire' (yangın) farkını anlaması için hipotez şablonu ekledik
    hypothesis_template = "This emergency call is about {}."
    result = classifier(text, labels, hypothesis_template=hypothesis_template)
    
    return result['labels'][0]

def determine_unit(incident_type):
    """ Map incident type to the correct unit """
    if incident_type == "Medical Emergency":
        return "Ambulance / EMS"
    elif incident_type == "Fire or Smoke Incident":
        return "Fire Department"
    elif incident_type == "Armed Robbery / Shooting":
        return "Police (SWAT Required)"
    else:
        return "Police"

def extract_location(text):
    """ Extract GPE (Cities), FAC (Buildings), LOC (Locations) """
    doc = nlp_ner(text)
    locs = [ent.text for ent in doc.ents if ent.label_ in ['GPE', 'FAC', 'LOC', 'ORG']]
    return ", ".join(list(set(locs))) if locs else "Unknown Location"

def check_authenticity(text):
    labels = ["Genuine Emergency Call", "Prank or Fake Call", "Non-Emergency Inquiry"]
    result = classifier(text, labels)
    top_label = result['labels'][0]
    score = result['scores'][0]
    status = top_label
    if score < 0.60:
        status += " (Low Confidence)"
    return status, score

def analyze_emotion(text):
    try:
        res = emotion_analyzer(text[:512])[0]
        return res['label'], res['score']
    except:
        return "Unknown", 0.0

# ==========================================
# 3. MAIN PIPELINE
# ==========================================
def main():
    input_file = '../data/labels/auto_transcripts.csv'
    
    try:
        df = pd.read_csv(input_file)
        print(f"Data Loaded: {len(df)} rows found.")
    except FileNotFoundError:
        print("File not found.")
        return

    results = []
    
    for index, row in df.head(50).iterrows():
        text = str(row['text'])
        
        # Geliştirilmiş analizler
        inc_type = analyze_incident_type(text)
        unit = determine_unit(inc_type)
        loc = extract_location(text)
        authenticity, auth_score = check_authenticity(text)
        emotion, emo_score = analyze_emotion(text)
        
        print(f"[{index+1}] Path: {row['path'][:30]}... | Type: {inc_type} | Unit: {unit}")
        
        results.append({
            "Path": row['path'],
            "Detected_Location": loc,
            "Incident_Type": inc_type,
            "Required_Unit": unit,
            "Call_Authenticity": authenticity,
            "Dominant_Emotion": emotion
        })

    output_df = pd.DataFrame(results)
    output_df.to_csv("refined_incident_analysis.csv", index=False)
    print("\nAnalysis Complete! 'refined_incident_analysis.csv' saved.")

if __name__ == "__main__":
    main()

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading AI Models... (This might take a minute)


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2469.09it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Data Loaded: 706 rows found.
[1] Path: data/raw/911_recordings/audio/... | Type: Armed Robbery / Shooting | Unit: Police (SWAT Required)
[2] Path: data/raw/911_recordings/audio/... | Type: Suspicious Activity | Unit: Police
[3] Path: data/raw/911_recordings/audio/... | Type: Suspicious Activity | Unit: Police
[4] Path: data/raw/911_recordings/audio/... | Type: Domestic Violence | Unit: Police
[5] Path: data/raw/911_recordings/audio/... | Type: Suspicious Activity | Unit: Police
[6] Path: data/raw/911_recordings/audio/... | Type: Medical Emergency | Unit: Ambulance / EMS
[7] Path: data/raw/911_recordings/audio/... | Type: Medical Emergency | Unit: Ambulance / EMS
[8] Path: data/raw/911_recordings/audio/... | Type: Suspicious Activity | Unit: Police
[9] Path: data/raw/911_recordings/audio/... | Type: Domestic Violence | Unit: Police
[10] Path: data/raw/911_recordings/audio/... | Type: Suspicious Activity | Unit: Police
[11] Path: data/raw/911_recordings/audio/... | Type: Fire or Smoke In